# 1D generation–diffusion–recombination solver — release 0.3

This notebook connects the bulk and optical modules. It solves the steady-state
excess-carrier continuity equation

$$\frac{dJ}{dx}=G(x)-R(\Delta n),\qquad
J=-D_a(\Delta n)\frac{d\Delta n}{dx}$$

with independent front and rear surface recombination velocities. The local
$D_a$, radiative, Auger, and bulk-SRH rates come from the Fermi–Dirac bulk model.

The result is a prediction of the carrier profile and **internal radiative
recombination flux**, not yet the detected PL signal: photon recycling,
reabsorption, collection optics, and surface band bending are not included.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "src").exists(): root = root.parent
sys.path.insert(0, str(root / "src"))
from surpass import (BulkModel, BulkTransportTable, Layer, OpticalStack,
                        SteadyState1DSolver, Wafer, get_optical_material)

try:
    import scienceplots
    plt.style.use(["science", "notebook", "no-latex"])
except ImportError:
    plt.style.use("default")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Editable experiment and numerical grid

The near-surface grid is essential: at 514 nm the demonstration InP absorption
depth is only about 0.09 µm, versus a 620 µm wafer. The optical module supplies
the **exact cell-averaged** Beer–Lambert generation, avoiding sampling error in
the steep first few cells.

In [ ]:
wavelength_nm = 514.0
thickness_um = 620.0
thickness_cm = thickness_um*1e-4
pox_nm, alox_nm = 10.0, 10.0
tau_bulk_srh_s = 100e-9       # assumption, not a material constant
enable_bgn = False
rear_s_default = 1e5          # cm/s; deliberately pessimistic starting scenario
front_s_values = [10.0, 1e3, 1e5]

# Intensities from the current experimental comparison; keep editable.
wafer_settings = {
    "n-InP": dict(wafer=Wafer("InP", donor_cm3=8.5e18, thickness_um=thickness_um),
                  intensity=1.0, color="tab:blue"),
    "p-InP": dict(wafer=Wafer("InP", acceptor_cm3=5.1e18, thickness_um=thickness_um),
                  intensity=13.0, color="tab:red"),
}

air = get_optical_material("air")
inp = get_optical_material("InP_demo")
pox = get_optical_material("POx_demo")
alox = get_optical_material("AlOx_demo")
stack = OpticalStack(air, [Layer(pox, pox_nm, "POx"), Layer(alox, alox_nm, "AlOx")], inp)

# 0-5 um: logarithmically refined; 5-620 um: progressively coarse.
edges_cm = np.r_[0.0, np.geomspace(1e-8, 5e-4, 140),
                 np.linspace(5e-4, thickness_cm, 161)[1:]]
centres_um = 0.5*(edges_cm[:-1] + edges_cm[1:])*1e4
print(f"Finite-volume cells: {len(centres_um)}; first/last widths = "
      f"{np.diff(edges_cm)[0]*1e7:.3g}/{np.diff(edges_cm)[-1]*1e4:.3g} nm/um")
print(f"Optical 1/e depth = {stack.solve(wavelength_nm).absorption_depth_nm:.2f} nm")

## 2. Build injection-dependent transport tables

In [ ]:
solvers, generations = {}, {}
for label, settings in wafer_settings.items():
    bulk = BulkModel(settings["wafer"], enable_bgn=enable_bgn, tau_srh_s=tau_bulk_srh_s)
    table = BulkTransportTable.from_bulk_model(
        bulk, delta_min_cm3=1e5, delta_max_cm3=1e21, points=241
    )
    solvers[label] = SteadyState1DSolver(table)
    generations[label] = stack.generation_cell_average_cm3_s(
        wavelength_nm, settings["intensity"], edges_cm
    )

fig, ax = plt.subplots(figsize=(6, 3.5))
for label, generation in generations.items():
    ax.loglog(centres_um, generation, label=label)
ax.axvline(stack.solve(wavelength_nm).absorption_depth_nm/1000, color="0.4", ls="--",
           label="optical 1/e depth")
ax.set(xlabel="Depth (µm)", ylabel="$G$ (cm$^{-3}$ s$^{-1}$)")
ax.legend(fontsize=8); fig.tight_layout()

## 3. Front-surface passivation sweep

Here $S_f$ is phenomenological. A later electrostatic/interface-SRH module will
calculate an injection-dependent boundary rate from $D_{it}$, capture cross
sections, and fixed charge instead of prescribing one constant velocity.

In [ ]:
results, rows = {}, []
for label, settings in wafer_settings.items():
    for front_s in front_s_values:
        result = solvers[label].solve(
            thickness_cm, generations[label], front_s, rear_s_default,
            cell_edges_cm=edges_cm, max_nfev=1200,
        )
        results[(label, front_s)] = result
        rows.append({
            "wafer": label, "intensity (W/cm2)": settings["intensity"],
            "Sf (cm/s)": front_s, "Sr (cm/s)": rear_s_default,
            "<dn> (cm^-3)": result.average_delta_n_cm3,
            "tau_eff (ns)": 1e9*result.effective_lifetime_s,
            "front fraction": result.front_surface_recombination_cm2_s/result.generated_flux_cm2_s,
            "rear fraction": result.rear_surface_recombination_cm2_s/result.generated_flux_cm2_s,
            "bulk rad fraction": result.internal_radiative_yield,
            "bulk Auger fraction": result.bulk_auger_recombination_cm2_s/result.generated_flux_cm2_s,
            "bulk SRH fraction": result.bulk_srh_recombination_cm2_s/result.generated_flux_cm2_s,
            "internal rad flux (cm^-2 s^-1)": result.bulk_radiative_recombination_cm2_s,
            "balance error": result.relative_balance_error,
            "max cell residual": result.maximum_scaled_cell_residual,
            "converged": result.converged,
        })
summary = pd.DataFrame(rows)
display(summary.style.format({
    "<dn> (cm^-3)": "{:.3e}", "tau_eff (ns)": "{:.3g}",
    "front fraction": "{:.3f}", "rear fraction": "{:.3f}",
    "bulk rad fraction": "{:.3f}", "bulk Auger fraction": "{:.3f}",
    "bulk SRH fraction": "{:.3f}", "internal rad flux (cm^-2 s^-1)": "{:.3e}",
    "balance error": "{:.2e}", "max cell residual": "{:.2e}"}))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for row, (label, settings) in enumerate(wafer_settings.items()):
    for front_s in front_s_values:
        result = results[(label, front_s)]
        axes[row, 0].semilogx(result.depth_cm*1e4, result.delta_n_cm3,
                              label=f"$S_f$={front_s:g} cm/s")
        axes[row, 1].plot(result.depth_cm*1e4, result.delta_n_cm3,
                         label=f"$S_f$={front_s:g} cm/s")
    axes[row, 0].set(title=f"{label}: near front", xlim=(1e-4, 20),
                     xlabel="Depth (µm)", ylabel="$\\Delta n$ (cm$^{-3}$)")
    axes[row, 1].set(title=f"{label}: full wafer", xlim=(0, thickness_um),
                     xlabel="Depth (µm)", ylabel="$\\Delta n$ (cm$^{-3}$)")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()

## 4. Does the rear surface matter?

In [ ]:
rear_s_values = [0.0, 1e2, 1e4, 1e6]
front_s_reference = 1e3
rear_rows = []
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (label, settings) in zip(axes, wafer_settings.items()):
    reference_flux = None
    for rear_s in rear_s_values:
        result = solvers[label].solve(
            thickness_cm, generations[label], front_s_reference, rear_s,
            cell_edges_cm=edges_cm, max_nfev=1200,
        )
        if reference_flux is None: reference_flux = result.bulk_radiative_recombination_cm2_s
        rear_rows.append({"wafer": label, "Sr (cm/s)": rear_s,
                          "tau_eff (ns)": 1e9*result.effective_lifetime_s,
                          "internal rad flux / reflecting rear":
                              result.bulk_radiative_recombination_cm2_s/reference_flux,
                          "rear recombination fraction":
                              result.rear_surface_recombination_cm2_s/result.generated_flux_cm2_s})
        ax.plot(result.depth_cm*1e4, result.delta_n_cm3, label=f"$S_r$={rear_s:g}")
    ax.set(title=label, xlabel="Depth (µm)", ylabel="$\\Delta n$ (cm$^{-3}$)")
    ax.legend(fontsize=8)
fig.tight_layout()
display(pd.DataFrame(rear_rows).style.format(precision=4))

## Interpretation and limitations

- The injection level is now an **output** of absorbed pump flux, bulk
  recombination, diffusion, wafer thickness, and the two boundary conditions;
  it is no longer chosen arbitrarily as in a $J_0$ normalization.
- The calculated internal radiative flux is the cleanest first proxy for PL.
  Multiplying it by a fixed escape factor is only qualitative because photon
  recycling makes escape and bulk recombination mutually coupled.
- Whether the rear matters is answered quantitatively by the $S_r$ sweep. A
  shallow generation profile does not by itself make the rear irrelevant when
  the diffusion length is comparable to the wafer thickness.
- Constant $S_f$ cannot yet represent field-effect passivation. The next
  release should couple the surface-potential/interface-SRH calculation to the
  boundary flux, ideally as $U_s(\Delta n_s,Q_f,D_{it},\sigma_n,\sigma_p)$.
- TRPL requires the time-dependent form of the same finite-volume system and a
  model for the detected photon flux; it should follow after the steady-state
  boundary model is validated.